In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/nifty-50-20072021/NSEI .csv


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from catboost import CatBoostRegressor

# =====================================================
# LOAD DATA
# =====================================================

test = pd.read_csv("/kaggle/input/datasets/piusdutta/flipkart/test.csv")
train = pd.read_csv("/kaggle/input/datasets/piusdutta/flipkart/train.csv")

test_ids = test["Index"]

# =====================================================
# MISSING VALUES
# =====================================================

cat_cols = [
    'RoadType',
    'Weather',
    'LargeVehicles',
    'Landmarks'
]

for col in cat_cols:
    train[col] = train[col].fillna("Missing")
    test[col] = test[col].fillna("Missing")

train["Temperature"] = train["Temperature"].fillna(
    train["Temperature"].median()
)

test["Temperature"] = test["Temperature"].fillna(
    train["Temperature"].median()
)

# =====================================================
# FEATURE ENGINEERING FUNCTION
# =====================================================

def create_features(df):

    df = df.copy()

    dt = pd.to_datetime(
        df["timestamp"],
        format="%H:%M"
    )

    df["hour"] = dt.dt.hour
    df["minute"] = dt.dt.minute

    df["time_minutes"] = (
        df["hour"] * 60 +
        df["minute"]
    )

    df["time_sin"] = np.sin(
        2 * np.pi * df["time_minutes"] / 1440
    )

    df["time_cos"] = np.cos(
        2 * np.pi * df["time_minutes"] / 1440
    )

    df["hour_sin"] = np.sin(
        2 * np.pi * df["hour"] / 24
    )

    df["hour_cos"] = np.cos(
        2 * np.pi * df["hour"] / 24
    )

    df["day"] = df["day"].astype(str)

    df["is_morning_peak"] = (
        df["hour"].between(7, 10)
    ).astype(int)

    df["is_evening_peak"] = (
        df["hour"].between(17, 20)
    ).astype(int)

    df["geo_weather"] = (
        df["geohash"].astype(str)
        + "_"
        + df["Weather"].astype(str)
    )

    df["geo_road"] = (
        df["geohash"].astype(str)
        + "_"
        + df["RoadType"].astype(str)
    )

    df["temp_bin"] = pd.cut(
        df["Temperature"],
        bins=[-100,10,20,30,40,100],
        labels=[
            "very_cold",
            "cold",
            "mild",
            "warm",
            "hot"
        ]
    ).astype(str)

    df["temp_x_hour"] = (
        df["Temperature"]
        * df["hour"]
    )

    return df

# =====================================================
# CREATE FEATURES
# =====================================================

train = create_features(train)
test = create_features(test)

# =====================================================
# GEOHASH FREQUENCY
# =====================================================

geo_freq = train["geohash"].value_counts()

train["geohash_freq"] = (
    train["geohash"]
    .map(geo_freq)
)

test["geohash_freq"] = (
    test["geohash"]
    .map(geo_freq)
    .fillna(0)
)

# =====================================================
# DROP UNUSED
# =====================================================

train.drop(
    columns=["timestamp"],
    inplace=True
)

test.drop(
    columns=["timestamp"],
    inplace=True
)

# =====================================================
# TARGET
# =====================================================

TARGET = "demand"

X = train.drop(['Index',TARGET], axis=1)
y = train[TARGET]

test_df = test.drop(
    columns=["Index"],
    errors="ignore"
)

# =====================================================
# CATEGORICAL FEATURES
# =====================================================

cat_features = [
    'geohash',
    'day',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'geo_weather',
    'geo_road',
    'temp_bin'
]

# =====================================================
# 5 FOLD CATBOOST
# =====================================================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof = np.zeros(len(X))
test_preds = np.zeros(len(test_df))

for fold, (trn_idx, val_idx) in enumerate(kf.split(X)):

    print(f"\nFold {fold+1}")

    X_train = X.iloc[trn_idx]
    X_valid = X.iloc[val_idx]

    y_train = y.iloc[trn_idx]
    y_valid = y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=12000,
        learning_rate=0.02,
        depth=10,
        l2_leaf_reg=5,
        loss_function='RMSE',
        random_seed=42 + fold,
        eval_metric='R2',
        task_type='CPU',
        verbose=500
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
        early_stopping_rounds=1000
    )

    oof[val_idx] = model.predict(X_valid)

    test_preds += (
        model.predict(test_df)
        / 5
    )

# =====================================================
# SUBMISSION
# =====================================================

submission = pd.DataFrame({
    "Index": test_ids,
    "demand": test_preds
})

submission.to_csv(
    "sample_submission.csv",
    index=False
)

print("\nSaved submission.csv")
print(submission.head())


Fold 1
0:	learn: 0.0312014	test: 0.0319450	best: 0.0319450 (0)	total: 157ms	remaining: 31m 24s
500:	learn: 0.9326575	test: 0.9271711	best: 0.9271712 (498)	total: 37.1s	remaining: 14m 12s
1000:	learn: 0.9463283	test: 0.9359474	best: 0.9359474 (1000)	total: 1m 16s	remaining: 13m 58s
1500:	learn: 0.9552144	test: 0.9404781	best: 0.9404781 (1498)	total: 1m 57s	remaining: 13m 40s
2000:	learn: 0.9610525	test: 0.9430371	best: 0.9430371 (2000)	total: 2m 38s	remaining: 13m 10s
2500:	learn: 0.9652153	test: 0.9446096	best: 0.9446107 (2499)	total: 3m 19s	remaining: 12m 38s
3000:	learn: 0.9682278	test: 0.9455218	best: 0.9455218 (3000)	total: 4m 2s	remaining: 12m 6s
3500:	learn: 0.9707134	test: 0.9462024	best: 0.9462025 (3499)	total: 4m 44s	remaining: 11m 30s
4000:	learn: 0.9727423	test: 0.9466410	best: 0.9466417 (3994)	total: 5m 25s	remaining: 10m 51s
4500:	learn: 0.9745940	test: 0.9470334	best: 0.9470334 (4500)	total: 6m 7s	remaining: 10m 11s
5000:	learn: 0.9761713	test: 0.9473138	best: 0.9473138 

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from catboost import CatBoostRegressor

# =====================================================
# LOAD DATA
# =====================================================

test = pd.read_csv("/kaggle/input/datasets/piusdutta/flipkart/test.csv")
train = pd.read_csv("/kaggle/input/datasets/piusdutta/flipkart/train.csv")

test_ids = test["Index"]

# =====================================================
# MISSING VALUES
# =====================================================

cat_cols = [
    'RoadType',
    'Weather',
    'LargeVehicles',
    'Landmarks'
]

for col in cat_cols:
    train[col] = train[col].fillna("Missing")
    test[col] = test[col].fillna("Missing")

train["Temperature"] = train["Temperature"].fillna(
    train["Temperature"].median()
)

test["Temperature"] = test["Temperature"].fillna(
    train["Temperature"].median()
)

# =====================================================
# FEATURE ENGINEERING FUNCTION
# =====================================================

def create_features(df):

    df = df.copy()

    dt = pd.to_datetime(
        df["timestamp"],
        format="%H:%M"
    )

    df["hour"] = dt.dt.hour
    df["minute"] = dt.dt.minute

    df["time_minutes"] = (
        df["hour"] * 60 +
        df["minute"]
    )

    df["time_sin"] = np.sin(
        2 * np.pi * df["time_minutes"] / 1440
    )

    df["time_cos"] = np.cos(
        2 * np.pi * df["time_minutes"] / 1440
    )

    df["hour_sin"] = np.sin(
        2 * np.pi * df["hour"] / 24
    )

    df["hour_cos"] = np.cos(
        2 * np.pi * df["hour"] / 24
    )

    df["day"] = df["day"].astype(str)

    df["is_morning_peak"] = (
        df["hour"].between(7, 10)
    ).astype(int)

    df["is_evening_peak"] = (
        df["hour"].between(17, 20)
    ).astype(int)

    df["geo_weather"] = (
        df["geohash"].astype(str)
        + "_"
        + df["Weather"].astype(str)
    )

    df["geo_road"] = (
        df["geohash"].astype(str)
        + "_"
        + df["RoadType"].astype(str)
    )

    df["temp_bin"] = pd.cut(
        df["Temperature"],
        bins=[-100,10,20,30,40,100],
        labels=[
            "very_cold",
            "cold",
            "mild",
            "warm",
            "hot"
        ]
    ).astype(str)

    df["temp_x_hour"] = (
        df["Temperature"]
        * df["hour"]
    )

    return df

# =====================================================
# CREATE FEATURES
# =====================================================

train = create_features(train)
test = create_features(test)

# =====================================================
# GEOHASH FREQUENCY
# =====================================================

geo_freq = train["geohash"].value_counts()

train["geohash_freq"] = (
    train["geohash"]
    .map(geo_freq)
)

test["geohash_freq"] = (
    test["geohash"]
    .map(geo_freq)
    .fillna(0)
)

# =====================================================
# DROP UNUSED
# =====================================================

train.drop(
    columns=["timestamp"],
    inplace=True
)

test.drop(
    columns=["timestamp"],
    inplace=True
)

# =====================================================
# TARGET
# =====================================================

TARGET = "demand"

X = train.drop(['Index',TARGET], axis=1)
y = train[TARGET]

test_df = test.drop(
    columns=["Index"],
    errors="ignore"
)

# =====================================================
# CATEGORICAL FEATURES
# =====================================================

cat_features = [
    'geohash',
    'day',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'geo_weather',
    'geo_road',
    'temp_bin'
]

# =====================================================
# 5 FOLD CATBOOST
# =====================================================

kf = KFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

oof = np.zeros(len(X))
test_preds = np.zeros(len(test_df))

for fold, (trn_idx, val_idx) in enumerate(kf.split(X)):

    print(f"\nFold {fold+1}")

    X_train = X.iloc[trn_idx]
    X_valid = X.iloc[val_idx]

    y_train = y.iloc[trn_idx]
    y_valid = y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=8000,
        learning_rate=0.02,
        depth=10,
        l2_leaf_reg=5,
        loss_function='RMSE',
        random_seed=42 + fold,
        eval_metric='R2',
        task_type='CPU',
        verbose=200
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
        early_stopping_rounds=1000
    )

    oof[val_idx] = model.predict(X_valid)

    test_preds += (
        model.predict(test_df)
        / 5
    )

# =====================================================
# SUBMISSION
# =====================================================

submission = pd.DataFrame({
    "Index": test_ids,
    "demand": test_preds
})

submission.to_csv(
    "sample_submission.csv",
    index=False
)

print("\nSaved submission.csv")
print(submission.head())


Fold 1
0:	learn: 0.0310282	test: 0.0313513	best: 0.0313513 (0)	total: 71.1ms	remaining: 9m 28s
200:	learn: 0.9073283	test: 0.9085473	best: 0.9085473 (200)	total: 12.2s	remaining: 7m 55s
400:	learn: 0.9252846	test: 0.9226535	best: 0.9226535 (400)	total: 25.5s	remaining: 8m 3s
600:	learn: 0.9335613	test: 0.9285303	best: 0.9285303 (600)	total: 38.2s	remaining: 7m 50s
800:	learn: 0.9389698	test: 0.9314156	best: 0.9314156 (800)	total: 51.8s	remaining: 7m 45s
1000:	learn: 0.9443639	test: 0.9345915	best: 0.9345915 (1000)	total: 1m 6s	remaining: 7m 42s
1200:	learn: 0.9485856	test: 0.9363787	best: 0.9363787 (1199)	total: 1m 21s	remaining: 7m 38s
1400:	learn: 0.9520818	test: 0.9381833	best: 0.9381833 (1400)	total: 1m 35s	remaining: 7m 30s
1600:	learn: 0.9548926	test: 0.9394334	best: 0.9394334 (1600)	total: 1m 50s	remaining: 7m 20s
1800:	learn: 0.9573630	test: 0.9403118	best: 0.9403118 (1800)	total: 2m 4s	remaining: 7m 9s
2000:	learn: 0.9595696	test: 0.9412302	best: 0.9412314 (1997)	total: 2m 19

In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from catboost import CatBoostRegressor

# =====================================================
# LOAD DATA
# =====================================================

test = pd.read_csv("/kaggle/input/datasets/piusdutta/flipkart/test.csv")
train = pd.read_csv("/kaggle/input/datasets/piusdutta/flipkart/train.csv")

test_ids = test["Index"]

# =====================================================
# MISSING VALUES
# =====================================================

cat_cols = [
    'RoadType',
    'Weather',
    'LargeVehicles',
    'Landmarks'
]

for col in cat_cols:
    train[col] = train[col].fillna("Missing")
    test[col] = test[col].fillna("Missing")

train["Temperature"] = train["Temperature"].fillna(
    train["Temperature"].median()
)

test["Temperature"] = test["Temperature"].fillna(
    train["Temperature"].median()
)

# =====================================================
# FEATURE ENGINEERING FUNCTION
# =====================================================

def create_features(df):

    df = df.copy()

    dt = pd.to_datetime(
        df["timestamp"],
        format="%H:%M"
    )

    df["hour"] = dt.dt.hour
    df["minute"] = dt.dt.minute

    df["time_minutes"] = (
        df["hour"] * 60 +
        df["minute"]
    )

    df["time_sin"] = np.sin(
        2 * np.pi * df["time_minutes"] / 1440
    )

    df["time_cos"] = np.cos(
        2 * np.pi * df["time_minutes"] / 1440
    )

    df["hour_sin"] = np.sin(
        2 * np.pi * df["hour"] / 24
    )

    df["hour_cos"] = np.cos(
        2 * np.pi * df["hour"] / 24
    )

    df["day"] = df["day"].astype(str)

    df["is_morning_peak"] = (
        df["hour"].between(7, 10)
    ).astype(int)

    df["is_evening_peak"] = (
        df["hour"].between(17, 20)
    ).astype(int)

    df["geo_weather"] = (
        df["geohash"].astype(str)
        + "_"
        + df["Weather"].astype(str)
    )

    df["geo_road"] = (
        df["geohash"].astype(str)
        + "_"
        + df["RoadType"].astype(str)
    )

    df["temp_bin"] = pd.cut(
        df["Temperature"],
        bins=[-100,10,20,30,40,100],
        labels=[
            "very_cold",
            "cold",
            "mild",
            "warm",
            "hot"
        ]
    ).astype(str)

    df["temp_x_hour"] = (
        df["Temperature"]
        * df["hour"]
    )

    return df

# =====================================================
# CREATE FEATURES
# =====================================================

train = create_features(train)
test = create_features(test)

# =====================================================
# GEOHASH FREQUENCY
# =====================================================

geo_freq = train["geohash"].value_counts()

train["geohash_freq"] = (
    train["geohash"]
    .map(geo_freq)
)

test["geohash_freq"] = (
    test["geohash"]
    .map(geo_freq)
    .fillna(0)
)

# =====================================================
# DROP UNUSED
# =====================================================

train.drop(
    columns=["timestamp"],
    inplace=True
)

test.drop(
    columns=["timestamp"],
    inplace=True
)

# =====================================================
# TARGET
# =====================================================

TARGET = "demand"

X = train.drop(['Index',TARGET], axis=1)
y = train[TARGET]

test_df = test.drop(
    columns=["Index"],
    errors="ignore"
)

# =====================================================
# CATEGORICAL FEATURES
# =====================================================

cat_features = [
    'geohash',
    'day',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'geo_weather',
    'geo_road',
    'temp_bin'
]

# =====================================================
# 5 FOLD CATBOOST
# =====================================================

# =====================================================
# SINGLE CATBOOST MODEL
# =====================================================

model = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.05,
    depth=3,
    l2_leaf_reg=5,
    loss_function='RMSE',
    eval_metric='R2',
    random_seed=42,
    task_type='CPU',
    verbose=200
)

model.fit(
    X,
    y,
    cat_features=cat_features
)

# =====================================================
# PREDICTIONS
# =====================================================

test_preds = model.predict(test_df)

# =====================================================
# SUBMISSION
# =====================================================

submission = pd.DataFrame({
    "Index": test_ids,
    "demand": test_preds
})

submission.to_csv(
    "sample_submission.csv",
    index=False
)

print("\nSaved sample_submission.csv")
print(submission.head())

0:	learn: 0.0754021	total: 21.6ms	remaining: 1m 47s
200:	learn: 0.8873675	total: 2.87s	remaining: 1m 8s
400:	learn: 0.8961206	total: 5.52s	remaining: 1m 3s
600:	learn: 0.9014489	total: 8.29s	remaining: 1m
800:	learn: 0.9051175	total: 11.1s	remaining: 58.1s
1000:	learn: 0.9076458	total: 13.9s	remaining: 55.5s
1200:	learn: 0.9096384	total: 16.8s	remaining: 53s
1400:	learn: 0.9111837	total: 19.6s	remaining: 50.3s
1600:	learn: 0.9126098	total: 22.4s	remaining: 47.6s
1800:	learn: 0.9141288	total: 25.3s	remaining: 45s
2000:	learn: 0.9154649	total: 28.2s	remaining: 42.2s
2200:	learn: 0.9165979	total: 31.2s	remaining: 39.7s
2400:	learn: 0.9175773	total: 34.2s	remaining: 37s
2600:	learn: 0.9186569	total: 37.2s	remaining: 34.3s
2800:	learn: 0.9195481	total: 40.2s	remaining: 31.5s
3000:	learn: 0.9201424	total: 43.1s	remaining: 28.7s
3200:	learn: 0.9209319	total: 46.1s	remaining: 25.9s
3400:	learn: 0.9218021	total: 49.1s	remaining: 23.1s
3600:	learn: 0.9224170	total: 52.2s	remaining: 20.3s
3800:	l